# 04 - Train a CNN on log-Mel spectrograms

This experiment trains a compact 2D CNN on log-Mel spectrograms. It uses the same official train and validation splits as the classical models. The test split is deliberately excluded from training and model selection.

The training sampler compensates for class imbalance by sampling classes approximately evenly. This does not change the original validation distribution.

## 1. Imports, paths and reproducibility

In [ ]:
from pathlib import Path
import sys, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.deep_learning import (
    SmallAudioCNN, make_loaders, train_cnn, evaluate_cnn, save_cnn_bundle
)

DATA_DIR = PROJECT_ROOT / 'notebooks' / 'data' / 'processed'
INDEX_PATH = DATA_DIR / 'audio_index.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if not INDEX_PATH.is_file():
    raise FileNotFoundError('Audio index missing. Run notebooks 01 and 02 first.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Index:', INDEX_PATH)

## 2. Load the split index and create DataLoaders

In [ ]:
audio_index = pd.read_csv(INDEX_PATH)
required = {'path', 'target', 'split'}
missing = required.difference(audio_index.columns)
if missing:
    raise ValueError(f'Audio index is missing columns: {sorted(missing)}')
if 'offset_seconds' not in audio_index.columns:
    audio_index['offset_seconds'] = 0.0
print('Rows:', len(audio_index))
display(audio_index.groupby(['split', 'target']).size().unstack(fill_value=0))

BATCH_SIZE = 64
train_loader, validation_loader, labels = make_loaders(
    audio_index, batch_size=BATCH_SIZE, sample_rate=16_000, n_mels=64
)
print('Classes:', labels)
print('Train batches:', len(train_loader), '| Validation batches:', len(validation_loader))

## 3. Inspect one log-Mel batch

In [ ]:
sample_features, sample_targets = next(iter(train_loader))
print('Batch shape:', tuple(sample_features.shape))
print('Target shape:', tuple(sample_targets.shape))
assert sample_features.ndim == 4 and sample_features.shape[1] == 1
assert torch.isfinite(sample_features).all()

plt.figure(figsize=(9, 4))
plt.imshow(sample_features[0, 0].numpy(), origin='lower', aspect='auto', cmap='magma')
plt.title(f'Example log-Mel spectrogram: {labels[int(sample_targets[0])]}')
plt.xlabel('Time frame')
plt.ylabel('Mel frequency bin')
plt.colorbar(label='Normalised log power')
plt.tight_layout()
plt.show()

## 4. Train the CNN

The default run uses the complete training split for six epochs. On a CPU this may take a while because spectrograms are calculated as batches are loaded. The validation split is evaluated after every epoch, and the weights with the best validation macro-F1 are retained.

In [ ]:
EPOCHS = 6
LEARNING_RATE = 1e-3
model = SmallAudioCNN(n_classes=len(labels))
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {parameter_count:,}')
training_started = time.perf_counter()
model, history = train_cnn(
    model, train_loader, validation_loader,
    epochs=EPOCHS, learning_rate=LEARNING_RATE, device=DEVICE
)
total_training_seconds = time.perf_counter() - training_started
print(f'Total training time: {total_training_seconds / 60:.1f} minutes')
display(history.round(4))

## 5. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['epoch'], history['train_loss'], marker='o', label='Train')
axes[0].plot(history['epoch'], history['validation_loss'], marker='o', label='Validation')
axes[0].set_title('CNN loss by epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss'); axes[0].legend()
axes[1].plot(history['epoch'], history['validation_macro_f1'], marker='o', label='Validation macro-F1')
axes[1].set_title('Validation macro-F1'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro-F1'); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluate on validation data only

In [ ]:
validation_result = evaluate_cnn(model, validation_loader, labels, device=DEVICE)
cnn_metrics = {key: validation_result[key] for key in [
    'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1'
 ]}
display(pd.DataFrame([cnn_metrics], index=['small_audio_cnn']).round(4))
display(pd.DataFrame(validation_result['report']).T.round(3))

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(
    confusion_matrix=validation_result['confusion_matrix'], display_labels=labels
 ).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
ax.set_title('CNN validation confusion matrix')
plt.tight_layout(); plt.show()

## 7. Compare with the classical baseline and save the CNN

In [ ]:
comparison_rows = [{'model': 'small_audio_cnn', **cnn_metrics}]
classical_metrics_path = MODEL_DIR / 'validation_model_comparison.json'
if classical_metrics_path.is_file():
    classical = pd.read_json(classical_metrics_path)
    for row in classical.to_dict(orient='records'):
        comparison_rows.append(row)
else:
    print('Classical metrics JSON not found; showing CNN metrics only.')
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))

checkpoint_path = MODEL_DIR / 'small_audio_cnn.pt'
save_cnn_bundle(model, labels, checkpoint_path, n_mels=64)
history_path = MODEL_DIR / 'cnn_training_history.csv'
history.to_csv(history_path, index=False)
metrics_path = MODEL_DIR / 'cnn_validation_model_comparison.json'
metrics_path.write_text(json.dumps(cnn_metrics, indent=2), encoding='utf-8')
print('Saved CNN checkpoint:', checkpoint_path)
print('Saved training history:', history_path)
print('Saved validation metrics:', metrics_path)
print('\nThe test split remains untouched. Do not report test performance until final model selection.')

## 8. Interpretation notes

- Compare macro-F1 and balanced accuracy across models because the class distribution is uneven.
- Accuracy alone can hide weak performance on smaller command classes.
- Inspect the confusion matrix to identify commands that the CNN confuses.
- The CNN receives time-frequency patterns, while the SVM and Random Forest receive summary MFCC statistics. The comparison therefore tests two different audio representations as well as different model families.
- The test split remains reserved for the final unbiased evaluation after choosing the model using validation results.